In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import random
import logging

# 關閉不必要的 transformers 警告訊息
logging.getLogger("transformers").setLevel(logging.ERROR)

print("⏳ 載入 Llama-3-Taiwan-8B 模型中 (啟用 4-bit 量化與強制 GPU 執行)，這可能需要幾分鐘下載...")
model_name = "yentinglin/Llama-3-Taiwan-8B-Instruct"

# 1. 設定 4-bit 量化參數
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

# 2. 載入 Tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name)

# 3. 載入模型 (強制放進 GPU 0)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map={"": 0}
)
print("✅ 載入完成！")
print("=" * 60)

def generate_advisor_question(target_dimension, conversation_history):
    # Llama 3 專用的 Prompt 格式
    prompt = f"""<|begin_of_text|><|start_header_id|>system<|end_header_id|>

你是一位專業、具備同理心的台灣量化理財顧問。你的目標是根據「待釐清的比較維度」，問使用者一個情境問題。

【嚴格限制，違反將導致系統崩潰】
1. 絕對不要列出選項（嚴禁出現 1. 2. 3. 或是 A. B. C. 等條列式結構）。
2. 只能說「一句話」，字數嚴格控制在 50 字以內。
3. 全程必須使用「繁體中文（台灣）」，用語要自然接地氣。
4. 只能問一個情境式的二選一問題，結尾必須是問號。
5. 根據對話歷史，不要重複使用者已經回答過的內容，並適時用白話文解釋專有名詞。

【完美範例】
待釐清維度：「歷史報酬 vs 抗波動」
顧問：您願意為了多賺一點潛在的報酬，承受短期內資產縮水兩成的風險嗎？

待釐清維度：「抗波動 vs 內扣成本」
顧問：為了讓資產在股災時比較抗跌，您願意每年多付出一點點的管理費給基金經理人嗎？<|eot_id|><|start_header_id|>user<|end_header_id|>

前置對話上下文：
{conversation_history}

待釐清維度：「{target_dimension}」
請生成一句顧問的問句：<|eot_id|><|start_header_id|>assistant<|end_header_id|>
"""
    
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs, 
            max_new_tokens=80, 
            temperature=0.4,
            repetition_penalty=1.15,
            pad_token_id=tokenizer.eos_token_id, # 避免 Llama 3 跳出 padding 警告
            do_sample=True,
            top_p=0.9
        )
    
    input_length = inputs.input_ids.shape[1]
    response = tokenizer.decode(outputs[0][input_length:], skip_special_tokens=True).strip()
    
    # 防呆切割：切掉模型可能產生的幻覺模板文字
    response = response.split("待釐清")[0].split("<|eot_id|>")[0].strip()
    return response

def run_interactive_simulation():
    dimensions = ["歷史報酬 vs 抗波動", "歷史報酬 vs 內扣成本", "抗波動 vs 內扣成本"]
    uncertainties = {dim: 1.0 for dim in dimensions}
    threshold = 0.3 
    
    conversation_history = "顧問：您好！我是您的專屬量化理財顧問。為了幫您配置最適合的投資組合，接下來我會問您幾個簡單的問題，了解一下您的投資想法。\n使用者：好的，沒問題。"
    
    print("\n 🤖 智能理財顧問對話測試開始 (輸入 'quit' 結束)")
    print("-" * 60)
    print("顧問：您好！我是您的專屬量化理財顧問。為了幫您配置最適合的投資組合，接下來我會問您幾個簡單的問題，了解一下您的投資想法。")
    print("使用者：好的，沒問題。\n")
    
    turn = 1
    while True:
        max_dim = max(uncertainties, key=uncertainties.get)
        max_uncert = uncertainties[max_dim]
        
        if max_uncert < threshold:
            print("\n🎉 [系統判定] 所有維度的不確定性皆已低於閾值！")
            print("顧問：非常感謝您的分享！我已經清楚了解您的投資偏好了。接下來，系統將為您計算專屬的最佳化投資組合。")
            break
            
        print(f"\n--- [內部狀態] 第 {turn} 輪 | 待釐清維度: {max_dim} (目前不確定性: {max_uncert:.2f}) ---")
        
        advisor_question = generate_advisor_question(max_dim, conversation_history)
        print(f"顧問：{advisor_question}")
        
        conversation_history += f"\n顧問：{advisor_question}"
        
        user_reply = input("，")
        if user_reply.lower() == 'quit':
            break
        print(f"使用者：{user_reply}")
        conversation_history += f"\n使用者：{user_reply}"
        
        drop_amount = random.uniform(0.2, 0.5)
        uncertainties[max_dim] = max(0.0, uncertainties[max_dim] - drop_amount)
        
        for dim in uncertainties:
            if dim != max_dim:
                uncertainties[dim] = max(0.0, uncertainties[dim] - random.uniform(0.0, 0.15))
        
        turn += 1
        
        max_uncert = max(uncertainties.values())
        if max_uncert >= threshold:
            print("⏳ 顧問正在思考下一個問題，請稍候...", end="\r", flush=True)

if __name__ == "__main__":
    run_interactive_simulation()

C:\Users\lojas\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\LocalCache\local-packages\Python310\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


⏳ 載入 Llama-3-Taiwan-8B 模型中 (啟用 4-bit 量化與強制 GPU 執行)，這可能需要幾分鐘下載...


C:\Users\lojas\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\LocalCache\local-packages\Python310\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\lojas\.cache\huggingface\hub\models--yentinglin--Llama-3-Taiwan-8B-Instruct. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading we

✅ 載入完成！

 🤖 智能理財顧問對話測試開始 (輸入 'quit' 結束)
------------------------------------------------------------
顧問：您好！我是您的專屬量化理財顧問。為了幫您配置最適合的投資組合，接下來我會問您幾個簡單的問題，了解一下您的投資想法。
使用者：好的，沒問題。


--- [內部狀態] 第 1 輪 | 待釐清維度: 歷史報酬 vs 抗波動 (目前不確定性: 1.00) ---
顧問：您更看重長期累積穩健報酬，還是希望在市場震盪時保護資產安全呢？
使用者：我希望能夠維持兩者間的平衡關係
⏳ 顧問正在思考下一個問題，請稍候...
--- [內部狀態] 第 2 輪 | 待釐清維度: 抗波動 vs 內扣成本 (目前不確定性: 1.00) ---
顧問：您是否願意增加一些內扣成本，以換取在市場劇烈波動時，資產獲得較佳的防護力？
使用者：我願意增加內扣成本，只要能夠守住資產本身
⏳ 顧問正在思考下一個問題，請稍候...
--- [內部狀態] 第 3 輪 | 待釐清維度: 歷史報酬 vs 內扣成本 (目前不確定性: 0.81) ---
顧問：面對高報酬通常伴隨著較高風險，您覺得可以接受多少額外的內扣成本，來降低市場大幅修正時資產遭受損失的可能性嗎？
使用者：我希望內扣成本能在不影響報酬的情況下越低越好
⏳ 顧問正在思考下一個問題，請稍候...
--- [內部狀態] 第 4 輪 | 待釐清維度: 歷史報酬 vs 抗波動 (目前不確定性: 0.67) ---
顧問：您認為，在追求優異投資報酬率與避免資產大幅虧損之間，有沒有辦法取得恰當的平衡點？
使用者：我認為可以
⏳ 顧問正在思考下一個問題，請稍候...
--- [內部狀態] 第 5 輪 | 待釐清維度: 歷史報酬 vs 內扣成本 (目前不確定性: 0.49) ---
顧問：您可否告訴我，如果提高內扣成本，可以讓您的投資組合在市場劇烈波動期間，獲得更多的保障程度？
使用者：我不太理解你的問題
⏳ 顧問正在思考下一個問題，請稍候...
--- [內部狀態] 第 6 輪 | 待釐清維度: 歷史報酬 vs 抗波動 (目前不確定性: 0.32) ---
顧問：如果提升內扣成本，可望使您的投資組合在市場劇烈波動時，擁有一定的抗性，那麼您願意支付這樣的成本嗎？